# 🏔️ Landslide Prediction Model — Sikkim, India

## Project Overview

This notebook develops a **binary classification model** to predict landslide susceptibility in Sikkim, India using satellite-derived geospatial features from Sentinel-1 (SAR) and Sentinel-2 (optical) imagery.

### Problem Statement
Landslides are among the most devastating natural hazards in mountainous regions. Early prediction of landslide-prone areas is critical for disaster risk reduction. This project uses 30 geospatial features — spectral indices, topographic parameters, hydrological attributes, and textural metrics — to classify terrain pixels as landslide-prone or stable.

### Dataset
- **Source**: Sikkim region, India
- **Size**: ~4 million pixel-level observations
- **Features**: 30 engineered geospatial features
- **Target**: `Decision` (0 = stable, 1 = landslide)
- **Challenge**: Severe class imbalance (~1.5% positive class)

### Approach
1. Stratified subsampling for tractability
2. Comprehensive EDA and preprocessing
3. Multiple model comparison (Logistic Regression → Random Forest → XGBoost → LightGBM)
4. Hyperparameter tuning with Optuna
5. Imbalance-aware evaluation (F1, ROC-AUC, PR-AUC)
6. Feature importance and explainability analysis

## 1. Imports and Configuration

In [1]:
# ============================================================
# Standard library
# ============================================================
import os
import sys
import warnings
import json
from pathlib import Path
from datetime import datetime

# ============================================================
# Data manipulation
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# Visualization
# ============================================================
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for file saving
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ============================================================
# Machine Learning
# ============================================================
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
import lightgbm as lgb

# ============================================================
# Hyperparameter Tuning
# ============================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# Imbalanced Learning
# ============================================================
from imblearn.over_sampling import SMOTE

# ============================================================
# Suppress warnings for cleaner output
# ============================================================
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("All imports successful.")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {__import__('sklearn').__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")

C:\Users\AGNIV GHOSH\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.
Python: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
NumPy: 2.3.5
Pandas: 2.3.1
Scikit-learn: 1.8.0
XGBoost: 3.2.0
LightGBM: 4.7.0


In [2]:
# ============================================================
# Configuration
# ============================================================
RANDOM_SEED = 42
SAMPLE_SIZE = 250_000         # Stratified subsample size
TEST_RATIO = 0.15             # Test split ratio
VAL_RATIO = 0.15              # Validation split ratio (from remaining)
CV_FOLDS = 5                  # Cross-validation folds
N_OPTUNA_TRIALS = 50          # Hyperparameter tuning trials

# Paths — relative to notebook location
PROJECT_ROOT = Path('..').resolve()
DATASET_PATH = PROJECT_ROOT / 'Sikkim' / 'Sikkim_dataset.csv'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
OUTPUT_DIR = PROJECT_ROOT / 'output'

# Create output directories
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
np.random.seed(RANDOM_SEED)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# Color palette for the project
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'positive': '#E84855',
    'negative': '#2E86AB',
    'accent': '#F18F01',
    'success': '#2ECC71',
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Project root: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware
Dataset path: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\Sikkim\Sikkim_dataset.csv
Dataset exists: True
Artifacts dir: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts
Output dir: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output


## 2. Dataset Loading

The full dataset contains ~4 million rows. To maintain tractability while preserving the full positive class, we:
1. Load the full dataset
2. Retain **all positive-class samples** (~60K)
3. Randomly undersample the negative class
4. Create a stratified subsample of ~250K rows

In [3]:
# ============================================================
# Load dataset with chunked reading for memory efficiency
# ============================================================
print("Loading dataset (this may take a moment for ~1.3GB file)...")
print(f"Reading from: {DATASET_PATH}")

# Read in chunks to get counts first
chunk_size = 500_000
positive_chunks = []
negative_chunks = []
total_rows = 0

for chunk in pd.read_csv(DATASET_PATH, chunksize=chunk_size):
    total_rows += len(chunk)
    pos = chunk[chunk['Decision'] == 1]
    neg = chunk[chunk['Decision'] == 0]
    positive_chunks.append(pos)
    negative_chunks.append(neg)
    print(f"  Processed {total_rows:,} rows... (pos so far: {sum(len(c) for c in positive_chunks):,})")

# Combine all positive samples
df_positive = pd.concat(positive_chunks, ignore_index=True)
df_negative = pd.concat(negative_chunks, ignore_index=True)

print(f"\nFull dataset: {total_rows:,} rows")
print(f"Positive class (landslide): {len(df_positive):,} ({len(df_positive)/total_rows*100:.2f}%)")
print(f"Negative class (stable): {len(df_negative):,} ({len(df_negative)/total_rows*100:.2f}%)")

Loading dataset (this may take a moment for ~1.3GB file)...
Reading from: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\Sikkim\Sikkim_dataset.csv


  Processed 500,000 rows... (pos so far: 8,938)


  Processed 1,000,000 rows... (pos so far: 14,947)


  Processed 1,500,000 rows... (pos so far: 17,012)


  Processed 2,000,000 rows... (pos so far: 20,055)


  Processed 2,500,000 rows... (pos so far: 32,553)


  Processed 3,000,000 rows... (pos so far: 47,441)


  Processed 3,500,000 rows... (pos so far: 50,812)


  Processed 4,000,000 rows... (pos so far: 56,691)



Full dataset: 4,000,000 rows
Positive class (landslide): 56,691 (1.42%)
Negative class (stable): 3,943,309 (98.58%)


In [4]:
# ============================================================
# Create stratified subsample
# ============================================================
# Keep ALL positive samples, undersample negative class
n_positive = len(df_positive)
n_negative_target = SAMPLE_SIZE - n_positive

if n_negative_target > len(df_negative):
    n_negative_target = len(df_negative)

df_negative_sampled = df_negative.sample(
    n=n_negative_target, random_state=RANDOM_SEED
)

df = pd.concat([df_positive, df_negative_sampled], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)  # Shuffle

print(f"Subsample size: {len(df):,}")
print(f"Positive: {df['Decision'].sum():,} ({df['Decision'].mean()*100:.2f}%)")
print(f"Negative: {(df['Decision']==0).sum():,} ({(1-df['Decision'].mean())*100:.2f}%)")
print(f"\nImbalance ratio: 1:{(df['Decision']==0).sum() // df['Decision'].sum()}")

# Free memory
del positive_chunks, negative_chunks, df_positive, df_negative, df_negative_sampled
import gc; gc.collect()

Subsample size: 250,000
Positive: 56,691 (22.68%)
Negative: 193,309 (77.32%)

Imbalance ratio: 1:3


0

## 3. Dataset Inspection

In [5]:
# ============================================================
# Basic structure
# ============================================================
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

print("Column Names and Types:")
print("-" * 40)
for col in df.columns:
    print(f"  {col:25s}  {str(df[col].dtype):10s}  unique={df[col].nunique():>10,}")

DATASET OVERVIEW
Shape: (250000, 33)
Rows: 250,000
Columns: 33

Memory usage: 66.0 MB

Column Names and Types:
----------------------------------------
  latitude                   float64     unique=   242,498
  longitude                  float64     unique=   242,498
  ARVI                       float64     unique=   247,355
  ASM                        float64     unique=    15,791
  BSI                        float64     unique=   248,505
  Contrast                   float64     unique=     1,146
  Dissimilarity              float64     unique=       667
  EVI                        float64     unique=   249,499
  Energy                     float64     unique=    15,789
  Entropy                    float64     unique=    62,610
  GLCMCorrelation            float64     unique=    78,996
  GLCMMean                   float64     unique=    13,035
  GNDVI                      float64     unique=   248,254
  GRVI                       float64     unique=   247,397
  Homogeneity         

  curvature                  float64     unique=   219,027
  elevation                  float64     unique=   217,499
  fdr                        int64       unique=         9
  mNDMI                      float64     unique=   248,945
  mNDWI                      float64     unique=   248,372
  plan_curvature             float64     unique=   219,224
  profile_curvature          float64     unique=   219,179
  slope                      float64     unique=   217,829
  spi                        float64     unique=   219,063
  tri                        float64     unique=   218,192
  twi                        float64     unique=   218,202
  Decision                   int64       unique=         2


In [6]:
# ============================================================
# Feature categories (from feature_list.csv)
# ============================================================
FEATURE_CATEGORIES = {
    'Spectral': ['ARVI', 'BSI', 'EVI', 'GNDVI', 'GRVI', 'MSAVI', 
                 'NDTI', 'NDVI', 'NDWI', 'SAVI', 'mNDMI', 'mNDWI'],
    'Topographical': ['aspect', 'curvature', 'elevation', 'plan_curvature', 
                      'profile_curvature', 'slope', 'tri', 'twi'],
    'Hydrological': ['fdr', 'spi'],
    'Textural': ['ASM', 'Contrast', 'Dissimilarity', 'Energy', 'Entropy', 
                 'GLCMCorrelation', 'GLCMMean', 'Homogeneity'],
}

COORDINATE_COLS = ['latitude', 'longitude']
TARGET_COL = 'Decision'

ALL_FEATURES = [f for feats in FEATURE_CATEGORIES.values() for f in feats]
print(f"Total features: {len(ALL_FEATURES)}")
for cat, feats in FEATURE_CATEGORIES.items():
    print(f"  {cat}: {len(feats)} features → {feats}")

print(f"\nCoordinates: {COORDINATE_COLS}")
print(f"Target: {TARGET_COL}")

Total features: 30
  Spectral: 12 features → ['ARVI', 'BSI', 'EVI', 'GNDVI', 'GRVI', 'MSAVI', 'NDTI', 'NDVI', 'NDWI', 'SAVI', 'mNDMI', 'mNDWI']
  Topographical: 8 features → ['aspect', 'curvature', 'elevation', 'plan_curvature', 'profile_curvature', 'slope', 'tri', 'twi']
  Hydrological: 2 features → ['fdr', 'spi']
  Textural: 8 features → ['ASM', 'Contrast', 'Dissimilarity', 'Energy', 'Entropy', 'GLCMCorrelation', 'GLCMMean', 'Homogeneity']

Coordinates: ['latitude', 'longitude']
Target: Decision


In [7]:
# ============================================================
# Statistical summary
# ============================================================
print("Statistical Summary (key percentiles):")
display(df[ALL_FEATURES + [TARGET_COL]].describe().T.round(4))

Statistical Summary (key percentiles):


,count,mean,std,min,25%,50%,75%,max
ARVI,250000.0,4.124000e-01,1.322000e-01,-1.225000e-01,0.3305,0.4208,0.5170,7.477000e-01
BSI,250000.0,-1.057000e-01,6.230000e-02,-3.128000e-01,-0.1515,-0.1125,-0.0684,1.744000e-01
EVI,250000.0,-2.833000e+00,1.413505e+02,-1.312884e+04,-4.5968,-1.7427,-0.7909,1.175643e+04
GNDVI,250000.0,2.656000e-01,1.120000e-01,-8.140000e-02,0.1839,0.2664,0.3537,5.886000e-01
GRVI,250000.0,5.720000e-02,2.920000e-02,-2.603000e-01,0.0460,0.0643,0.0765,1.385000e-01
MSAVI,250000.0,4.656000e-01,1.485000e-01,-3.300000e-02,0.3697,0.4790,0.5851,8.038000e-01
NDTI,250000.0,-4.940000e-02,4.100000e-02,-2.192000e-01,-0.0747,-0.0623,-0.0381,2.603000e-01
NDVI,250000.0,3.005000e-01,1.180000e-01,-1.620000e-02,0.2197,0.2887,0.3906,6.720000e-01
NDWI,250000.0,-2.656000e-01,1.120000e-01,-5.886000e-01,-0.3537,-0.2664,-0.1839,8.140000e-02
SAVI,250000.0,4.739000e-01,1.851000e-01,-2.430000e-02,0.3412,0.4735,0.6210,1.007900e+00


## 4. Data Quality Assessment

In [8]:
# ============================================================
# Missing values
# ============================================================
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(3)
}).sort_values('Missing Count', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0].to_string())

if missing.sum() == 0:
    print("\nNo missing values found!")
else:
    print(f"\nTotal cells with missing values: {missing.sum():,}")
    print(f"Columns with missing values: {(missing > 0).sum()}")

MISSING VALUE ANALYSIS
                 Missing Count  Missing %
GLCMCorrelation           3852      1.541

Total cells with missing values: 3,852
Columns with missing values: 1


In [9]:
# ============================================================
# Duplicate detection
# ============================================================
print("=" * 60)
print("DUPLICATE ANALYSIS")
print("=" * 60)

n_duplicates = df.duplicated().sum()
n_duplicates_features = df[ALL_FEATURES].duplicated().sum()

print(f"Exact duplicate rows: {n_duplicates:,} ({n_duplicates/len(df)*100:.2f}%)")
print(f"Duplicate feature vectors (ignoring target): {n_duplicates_features:,}")

if n_duplicates > 0:
    print("\nNote: Some duplicates are expected in pixel-level geospatial data")
    print("(adjacent pixels can have identical feature values)")

DUPLICATE ANALYSIS


Exact duplicate rows: 0 (0.00%)
Duplicate feature vectors (ignoring target): 0


In [10]:
# ============================================================
# Outlier detection (IQR method)
# ============================================================
print("=" * 60)
print("OUTLIER ANALYSIS (IQR method, >3× IQR)")
print("=" * 60)

outlier_summary = []
for col in ALL_FEATURES:
    if df[col].dtype in ['float64', 'int64']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        if n_outliers > 0:
            outlier_summary.append({
                'Feature': col,
                'Outliers': n_outliers,
                'Outlier %': round(n_outliers / len(df) * 100, 3),
                'Min': df[col].min(),
                'Max': df[col].max(),
                'IQR Lower': round(lower, 4),
                'IQR Upper': round(upper, 4),
            })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
print(outlier_df.to_string(index=False))
print(f"\nFeatures with outliers: {len(outlier_df)}/{len(ALL_FEATURES)}")

OUTLIER ANALYSIS (IQR method, >3× IQR)


          Feature  Outliers  Outlier %           Min          Max  IQR Lower  IQR Upper
              EVI     33590     13.436 -1.312884e+04 1.175643e+04   -16.0144    10.6267
  GLCMCorrelation     27683     11.073 -5.551115e-17 1.000000e+00     0.9757     1.0176
              spi     23214      9.286 -6.013407e+18 4.060464e+19  -944.7010  1425.9834
              fdr     21266      8.506  0.000000e+00 1.280000e+02   -80.0000   116.0000
        curvature     11399      4.560  1.007104e-08 3.337550e-02    -0.0006     0.0009
             NDTI     10141      4.056 -2.191783e-01 2.603056e-01    -0.1845     0.0717
   plan_curvature      4312      1.725 -4.902790e+00 2.390460e+00    -0.0528     0.0529
         Contrast      4127      1.651  0.000000e+00 2.248571e+01    -1.5286     3.3714
profile_curvature      2128      0.851 -6.848983e-02 6.372305e-02    -0.0195     0.0195
             GRVI      1524      0.610 -2.603056e-01 1.385062e-01    -0.0455     0.1679
              twi      1505     

## 5. Exploratory Data Analysis

Comprehensive visualization of dataset characteristics, feature distributions, correlations, and target-feature relationships.

In [11]:
# ============================================================
# 5.1 Target Variable Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df[TARGET_COL].value_counts()
labels = ['Stable (0)', 'Landslide (1)']
colors = [COLORS['negative'], COLORS['positive']]

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (v, pct) in enumerate(zip(counts.values, counts.values / len(df) * 100)):
    axes[0].text(i, v + len(df)*0.005, f'{v:,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, explode=(0, 0.1), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Severe Class Imbalance in Landslide Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'target_distribution.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'target_distribution.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\target_distribution.png


In [12]:
# ============================================================
# 5.2 Missing Value Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Missing count by column
missing_counts = df[ALL_FEATURES].isnull().sum().sort_values(ascending=False)
cols_with_missing = missing_counts[missing_counts > 0]

if len(cols_with_missing) > 0:
    axes[0].barh(cols_with_missing.index, cols_with_missing.values, color=COLORS['accent'])
    axes[0].set_title('Missing Values by Feature', fontweight='bold')
    axes[0].set_xlabel('Count')
    for i, v in enumerate(cols_with_missing.values):
        axes[0].text(v + 10, i, f'{v:,} ({v/len(df)*100:.2f}%)', va='center')
else:
    axes[0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center', fontsize=16)
    axes[0].set_title('Missing Values by Feature', fontweight='bold')

# Missing per class
if len(cols_with_missing) > 0:
    for label, color, name in [(0, COLORS['negative'], 'Stable'), (1, COLORS['positive'], 'Landslide')]:
        subset = df[df[TARGET_COL] == label]
        m = subset[cols_with_missing.index].isnull().sum()
        axes[1].barh([f"{c}" for c in m.index], m.values / len(subset) * 100,
                    alpha=0.7, label=name, color=color)
    axes[1].set_title('Missing Value % by Class', fontweight='bold')
    axes[1].set_xlabel('Percentage (%)')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No Missing Values', ha='center', va='center', fontsize=16)
    axes[1].set_title('Missing Values by Class', fontweight='bold')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'missing_values.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'missing_values.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\missing_values.png


In [13]:
# ============================================================
# 5.3 Feature Distributions (grouped by category)
# ============================================================
for cat_name, features in FEATURE_CATEGORIES.items():
    n_feats = len(features)
    n_cols = 4
    n_rows = (n_feats + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.5 * n_rows))
    axes = axes.flatten() if n_rows > 1 else (axes if n_feats > 1 else [axes])
    
    for i, feat in enumerate(features):
        ax = axes[i]
        for label, color, name in [(0, COLORS['negative'], 'Stable'), 
                                    (1, COLORS['positive'], 'Landslide')]:
            data = df[df[TARGET_COL] == label][feat].dropna()
            # Clip extreme values for visualization
            q_low, q_high = data.quantile(0.01), data.quantile(0.99)
            data_clipped = data.clip(q_low, q_high)
            ax.hist(data_clipped, bins=50, alpha=0.6, label=name, color=color, density=True)
        ax.set_title(feat, fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.tick_params(labelsize=8)
    
    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'{cat_name} Feature Distributions by Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fname = f'feature_distributions_{cat_name.lower()}.png'
    plt.savefig(ARTIFACTS_DIR / fname)
    plt.show()
    print(f"Saved: {ARTIFACTS_DIR / fname}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_distributions_spectral.png


Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_distributions_topographical.png


Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_distributions_hydrological.png


Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_distributions_textural.png


In [14]:
# ============================================================
# 5.4 Correlation Analysis
# ============================================================
fig, ax = plt.subplots(figsize=(18, 15))

corr_matrix = df[ALL_FEATURES + [TARGET_COL]].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': 'Correlation'},
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'correlation_matrix.png')
plt.show()

# Print high correlations with target
print("\nCorrelation with Target (Decision):")
print("-" * 40)
target_corr = corr_matrix[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
for feat, corr in target_corr.items():
    marker = "***" if abs(corr) > 0.15 else "**" if abs(corr) > 0.1 else "*" if abs(corr) > 0.05 else ""
    print(f"  {feat:25s}: {corr:+.4f}  {marker}")

print("\nHighly correlated feature pairs (|r| > 0.85):")
print("-" * 40)
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            print(f"  {corr_matrix.columns[i]:20s} ↔ {corr_matrix.columns[j]:20s}: {corr_matrix.iloc[i,j]:+.4f}")


Correlation with Target (Decision):
----------------------------------------
  mNDWI                    : -0.2875  ***
  GNDVI                    : +0.2130  ***
  NDWI                     : -0.2130  ***
  SAVI                     : +0.1560  ***
  Entropy                  : -0.1554  ***
  ASM                      : +0.1521  ***
  Energy                   : +0.1507  ***
  Homogeneity              : +0.1476  **
  Dissimilarity            : -0.1467  **
  GLCMMean                 : -0.1448  **
  MSAVI                    : +0.1421  **
  Contrast                 : -0.1338  **
  GRVI                     : -0.1281  **
  elevation                : -0.1012  **
  fdr                      : -0.0969  *
  aspect                   : -0.0865  *
  tri                      : -0.0534  *
  slope                    : -0.0509  *
  BSI                      : +0.0501  *
  NDVI                     : +0.0493  
  NDTI                     : +0.0401  
  curvature                : -0.0247  
  ARVI                  

In [15]:
# ============================================================
# 5.5 Outlier Analysis (Box plots for key features)
# ============================================================
# Select features with notable outliers
outlier_features = outlier_df.head(8)['Feature'].tolist() if len(outlier_df) > 0 else ALL_FEATURES[:8]

n_cols = 4
n_rows = (len(outlier_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(outlier_features):
    ax = axes[i]
    data_list = [df[df[TARGET_COL]==0][feat].dropna(), df[df[TARGET_COL]==1][feat].dropna()]
    bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
                    showfliers=True, flierprops={'markersize': 2, 'alpha': 0.3})
    bp['boxes'][0].set_facecolor(COLORS['negative'])
    bp['boxes'][1].set_facecolor(COLORS['positive'])
    for box in bp['boxes']:
        box.set_alpha(0.7)
    ax.set_title(feat, fontweight='bold')
    ax.tick_params(labelsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Outlier Analysis — Features with Most Extreme Values', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'outlier_analysis.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'outlier_analysis.png'}")

C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Lo

C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Local\Temp\ipykernel_10828\2043100703.py:15: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_list, labels=['Stable', 'Landslide'], patch_artist=True,
C:\Users\AGNIV GHOSH\AppData\Lo

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\outlier_analysis.png


In [16]:
# ============================================================
# 5.6 Feature-Target Relationships (top features by correlation)
# ============================================================
top_features = target_corr.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    for label, color, name in [(0, COLORS['negative'], 'Stable'),
                                (1, COLORS['positive'], 'Landslide')]:
        data = df[df[TARGET_COL] == label][feat].dropna()
        q_low, q_high = data.quantile(0.005), data.quantile(0.995)
        data_clipped = data.clip(q_low, q_high)
        ax.hist(data_clipped, bins=60, alpha=0.6, label=name, color=color, density=True)
    corr_val = target_corr[feat]
    ax.set_title(f'{feat}\n(r={corr_val:+.3f})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

plt.suptitle('Top 8 Features Most Correlated with Landslide Decision', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'feature_target_relationships.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'feature_target_relationships.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_target_relationships.png


In [17]:
# ============================================================
# 5.7 Feature Category Importance Overview
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))

category_corrs = {}
for cat, features in FEATURE_CATEGORIES.items():
    corrs = [abs(target_corr.get(f, 0)) for f in features]
    category_corrs[cat] = {
        'mean': np.mean(corrs),
        'max': np.max(corrs),
        'features': features
    }

cats = list(category_corrs.keys())
means = [category_corrs[c]['mean'] for c in cats]
maxs = [category_corrs[c]['max'] for c in cats]

x = np.arange(len(cats))
width = 0.35

bars1 = ax.bar(x - width/2, means, width, label='Mean |correlation|', 
               color=COLORS['primary'], alpha=0.8, edgecolor='white')
bars2 = ax.bar(x + width/2, maxs, width, label='Max |correlation|',
               color=COLORS['accent'], alpha=0.8, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=12)
ax.set_ylabel('|Correlation with Target|')
ax.set_title('Feature Category Correlation with Landslide Decision', fontsize=14, fontweight='bold')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'feature_category_analysis.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'feature_category_analysis.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\feature_category_analysis.png


## 6. Data Preprocessing

Robust preprocessing pipeline that:
1. Drops latitude/longitude (spatial data leakage)
2. Handles missing values (median imputation)
3. Clips extreme outliers (IQR-based)
4. All transformers fitted on training data only

In [18]:
# ============================================================
# 6.1 Prepare feature matrix and target
# ============================================================
X = df[ALL_FEATURES].copy()
y = df[TARGET_COL].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nDropped columns: {COORDINATE_COLS} (spatial data leakage prevention)")

Feature matrix shape: (250000, 30)
Target shape: (250000,)
Target distribution:
Decision
0    193309
1     56691
Name: count, dtype: int64

Dropped columns: ['latitude', 'longitude'] (spatial data leakage prevention)


In [19]:
# ============================================================
# 6.2 Handle missing values
# ============================================================
print("Missing values before imputation:")
missing_before = X.isnull().sum()
print(missing_before[missing_before > 0])

# We will impute after train/test split to prevent data leakage
# For now, just document what needs imputation
cols_with_missing = missing_before[missing_before > 0].index.tolist()
print(f"\nColumns requiring imputation: {cols_with_missing}")

Missing values before imputation:
GLCMCorrelation    3852
dtype: int64

Columns requiring imputation: ['GLCMCorrelation']


## 7. Feature Engineering

The dataset already contains well-engineered geospatial features. We add a few interaction features based on domain knowledge of landslide susceptibility factors.

In [20]:
# ============================================================
# 7.1 Domain-driven feature engineering
# ============================================================
# Slope × NDVI interaction — steep slopes with low vegetation are landslide-prone
X['slope_ndvi_interaction'] = X['slope'] * X['NDVI']

# Elevation × curvature — concave high-elevation terrain is susceptible
X['elevation_curvature'] = X['elevation'] * X['curvature']

# TWI × slope — wet steep slopes
X['twi_slope'] = X['twi'] * X['slope']

# Vegetation deficit (NDVI inverse as moisture stress indicator)
X['vegetation_stress'] = 1 - X['NDVI'].clip(0, 1)

NEW_FEATURES = ['slope_ndvi_interaction', 'elevation_curvature', 
                'twi_slope', 'vegetation_stress']

# Update feature list
ALL_FEATURES_ENGINEERED = ALL_FEATURES + NEW_FEATURES

print(f"Engineered {len(NEW_FEATURES)} new features:")
for f in NEW_FEATURES:
    print(f"  • {f}")
print(f"\nTotal features: {len(ALL_FEATURES_ENGINEERED)}")

Engineered 4 new features:
  • slope_ndvi_interaction
  • elevation_curvature
  • twi_slope
  • vegetation_stress

Total features: 34


## 8. Train / Validation / Test Split

Stratified splitting to preserve class ratios. All preprocessing fitted on training data only.

In [21]:
# ============================================================
# 8.1 Stratified split: 70% train / 15% val / 15% test
# ============================================================
# First split: train+val vs test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y
)

# Second split: train vs val (from remaining)
val_ratio_adjusted = VAL_RATIO / (1 - TEST_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_adjusted, random_state=RANDOM_SEED, stratify=y_temp
)

print("Split sizes:")
print(f"  Train: {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Val:   {X_val.shape[0]:,} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test:  {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.1f}%)")
print()
print("Class distribution per split:")
for name, split_y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pos = split_y.sum()
    total = len(split_y)
    print(f"  {name}: {pos:,} positive / {total-pos:,} negative ({pos/total*100:.2f}% positive)")

Split sizes:
  Train: 175,000 (70.0%)
  Val:   37,500 (15.0%)
  Test:  37,500 (15.0%)

Class distribution per split:
  Train: 39,683 positive / 135,317 negative (22.68% positive)
  Val: 8,504 positive / 28,996 negative (22.68% positive)
  Test: 8,504 positive / 28,996 negative (22.68% positive)


In [22]:
# ============================================================
# 8.2 Impute missing values (fitted on training data only)
# ============================================================
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train)

X_train = pd.DataFrame(imputer.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Missing values after imputation:")
print(f"  Train: {X_train.isnull().sum().sum()}")
print(f"  Val:   {X_val.isnull().sum().sum()}")
print(f"  Test:  {X_test.isnull().sum().sum()}")

Missing values after imputation:
  Train: 0
  Val:   0
  Test:  0


In [23]:
# ============================================================
# 8.3 Clip extreme outliers (fitted on training data)
# ============================================================
clip_bounds = {}
for col in X_train.columns:
    Q1 = X_train[col].quantile(0.01)
    Q3 = X_train[col].quantile(0.99)
    clip_bounds[col] = (Q1, Q3)

for split_X, name in [(X_train, 'Train'), (X_val, 'Val'), (X_test, 'Test')]:
    for col, (low, high) in clip_bounds.items():
        split_X[col] = split_X[col].clip(low, high)
    
print(f"Clipped {len(clip_bounds)} features to 1st-99th percentile range (fitted on training data)")

Clipped 34 features to 1st-99th percentile range (fitted on training data)


In [24]:
# ============================================================
# 8.4 Feature scaling (StandardScaler fitted on training data)
# ============================================================
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling complete. Training set statistics (should be ~0 mean, ~1 std):")
print(f"  Mean range: [{X_train_scaled.mean().min():.4f}, {X_train_scaled.mean().max():.4f}]")
print(f"  Std range:  [{X_train_scaled.std().min():.4f}, {X_train_scaled.std().max():.4f}]")

Scaling complete. Training set statistics (should be ~0 mean, ~1 std):
  Mean range: [-0.0000, 0.0000]


  Std range:  [1.0000, 1.0000]


In [25]:
# ============================================================
# 8.5 Handle class imbalance with SMOTE (training data only)
# ============================================================
print(f"Before SMOTE — Train: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=RANDOM_SEED, sampling_strategy=0.5)  # 1:2 ratio
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE  — Train: {pd.Series(y_train_smote).value_counts().to_dict()}")
print(f"\nSMOTE increased training set from {len(y_train):,} to {len(y_train_smote):,}")
print("Note: SMOTE applied only to training data. Validation and test sets are untouched.")

Before SMOTE — Train: {0: 135317, 1: 39683}


After SMOTE  — Train: {0: 135317, 1: 67658}

SMOTE increased training set from 175,000 to 202,975
Note: SMOTE applied only to training data. Validation and test sets are untouched.


## 9. Baseline Model

Logistic Regression as a simple, interpretable baseline to establish minimum performance benchmarks.

In [26]:
# ============================================================
# 9.1 Logistic Regression Baseline
# ============================================================
print("Training Logistic Regression baseline...")
print("=" * 60)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    solver='lbfgs',
    n_jobs=-1
)

lr_model.fit(X_train_smote, y_train_smote)

# Evaluate on validation set
y_val_pred_lr = lr_model.predict(X_val_scaled)
y_val_prob_lr = lr_model.predict_proba(X_val_scaled)[:, 1]

print("Logistic Regression — Validation Performance:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_lr):.4f}")
print(f"  F1-Score:  {f1_score(y_val, y_val_pred_lr):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_prob_lr):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_val, y_val_prob_lr):.4f}")

Training Logistic Regression baseline...


Logistic Regression — Validation Performance:
  Accuracy:  0.7657
  Precision: 0.4894
  Recall:    0.7694
  F1-Score:  0.5982
  ROC-AUC:   0.8374
  PR-AUC:    0.6291


## 10. Candidate Model Training

Training multiple candidate models and comparing their performance using stratified cross-validation.

In [27]:
# ============================================================
# 10.1 Define candidate models
# ============================================================
# Calculate scale_pos_weight for tree-based models
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"Scale pos weight: {scale_pos_weight:.1f}")

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED,
        eval_metric='logloss', verbosity=0, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1, num_leaves=63,
        scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED,
        verbose=-1, n_jobs=-1
    ),
}

print(f"Candidate models: {list(models.keys())}")

Scale pos weight: 3.4
Candidate models: ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']


In [28]:
# ============================================================
# 10.2 Train and evaluate all candidates
# ============================================================
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    print("-" * 40)
    
    # Use SMOTE data for Logistic Regression, original (unscaled) for trees
    if name == 'Logistic Regression':
        model.fit(X_train_smote, y_train_smote)
        y_pred = model.predict(X_val_scaled)
        y_prob = model.predict_proba(X_val_scaled)[:, 1]
    else:
        # Tree-based models work better with unscaled data + built-in class weights
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'Accuracy': accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1': f1_score(y_val, y_pred),
        'ROC-AUC': roc_auc_score(y_val, y_prob),
        'PR-AUC': average_precision_score(y_val, y_prob),
    }
    results[name] = metrics
    
    for metric, value in metrics.items():
        print(f"  {metric:12s}: {value:.4f}")

# Summary table
results_df = pd.DataFrame(results).T.sort_values('F1', ascending=False)
print("\n" + "=" * 60)
print("MODEL COMPARISON (sorted by F1)")
print("=" * 60)
display(results_df.round(4))


Training Logistic Regression...
----------------------------------------


  Accuracy    : 0.7657
  Precision   : 0.4894
  Recall      : 0.7694
  F1          : 0.5982
  ROC-AUC     : 0.8374
  PR-AUC      : 0.6291

Training Random Forest...
----------------------------------------


  Accuracy    : 0.9014
  Precision   : 0.7658
  Recall      : 0.8142
  F1          : 0.7893
  ROC-AUC     : 0.9522
  PR-AUC      : 0.8736

Training XGBoost...
----------------------------------------


  Accuracy    : 0.8934
  Precision   : 0.7129
  Recall      : 0.8869
  F1          : 0.7904
  ROC-AUC     : 0.9602
  PR-AUC      : 0.8829

Training LightGBM...
----------------------------------------


  Accuracy    : 0.9006
  Precision   : 0.7261
  Recall      : 0.9019
  F1          : 0.8045
  ROC-AUC     : 0.9667
  PR-AUC      : 0.8987

MODEL COMPARISON (sorted by F1)


,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
LightGBM,0.9006,0.7261,0.9019,0.8045,0.9667,0.8987
XGBoost,0.8934,0.7129,0.8869,0.7904,0.9602,0.8829
Random Forest,0.9014,0.7658,0.8142,0.7893,0.9522,0.8736
Logistic Regression,0.7657,0.4894,0.7694,0.5982,0.8374,0.6291


In [29]:
# ============================================================
# 10.3 Model comparison visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart comparison
metrics_to_plot = ['F1', 'ROC-AUC', 'PR-AUC', 'Recall']
x = np.arange(len(results_df))
width = 0.2
colors_bar = [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success']]

for i, metric in enumerate(metrics_to_plot):
    axes[0].bar(x + i * width, results_df[metric], width, label=metric, 
                color=colors_bar[i], alpha=0.85, edgecolor='white')

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(results_df.index, rotation=15, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Comparison — Key Metrics', fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1.05)

# ROC-AUC ranking
sorted_roc = results_df['ROC-AUC'].sort_values(ascending=True)
axes[1].barh(sorted_roc.index, sorted_roc.values, color=COLORS['primary'], alpha=0.8, edgecolor='white')
for i, v in enumerate(sorted_roc.values):
    axes[1].text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')
axes[1].set_title('ROC-AUC Ranking', fontweight='bold')
axes[1].set_xlim(0, 1.1)

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'model_comparison_validation.png')
plt.show()
print(f"Saved: {ARTIFACTS_DIR / 'model_comparison_validation.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\artifacts\model_comparison_validation.png


## 11. Hyperparameter Tuning

Using **Optuna** to tune the best-performing model from the comparison above.

In [30]:
# ============================================================
# 11.1 Select best model type for tuning
# ============================================================
best_model_name = results_df.index[0]  # Highest F1
print(f"Best model from comparison: {best_model_name}")
print(f"Tuning with Optuna ({N_OPTUNA_TRIALS} trials)...")
print()

# We'll tune XGBoost as primary and LightGBM as secondary if XGBoost wins
# If another model wins, we tune that instead

Best model from comparison: LightGBM
Tuning with Optuna (50 trials)...



In [31]:
# ============================================================
# 11.2 Optuna hyperparameter tuning
# ============================================================
def objective(trial):
    # Optuna objective for XGBoost/LightGBM tuning.
    if 'XGBoost' in best_model_name or 'Random Forest' in best_model_name:
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'scale_pos_weight': scale_pos_weight,
            'random_state': RANDOM_SEED,
            'eval_metric': 'logloss',
            'verbosity': 0,
            'n_jobs': -1,
        }
        model = xgb.XGBClassifier(**params)
    else:  # LightGBM
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'scale_pos_weight': scale_pos_weight,
            'random_state': RANDOM_SEED,
            'verbose': -1,
            'n_jobs': -1,
        }
        model = lgb.LGBMClassifier(**params)
    
    # Stratified cross-validation on training data
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    return scores.mean()

# Run optimization
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

print(f"\nBest trial F1 (CV): {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


Best trial F1 (CV): 0.8580

Best hyperparameters:
  n_estimators: 471
  max_depth: 15
  learning_rate: 0.2589898910477058
  num_leaves: 150
  subsample: 0.7874820875551369
  colsample_bytree: 0.9487283159585305
  min_child_samples: 25
  reg_alpha: 0.0020781973056227185
  reg_lambda: 0.008036141722925789


In [32]:
# ============================================================
# 11.3 Train final model with best hyperparameters
# ============================================================
best_params = study.best_params.copy()

if 'XGBoost' in best_model_name or 'Random Forest' in best_model_name:
    best_params.update({
        'scale_pos_weight': scale_pos_weight,
        'random_state': RANDOM_SEED,
        'eval_metric': 'logloss',
        'verbosity': 0,
        'n_jobs': -1,
    })
    final_model = xgb.XGBClassifier(**best_params)
    model_label = 'XGBoost (Tuned)'
else:
    best_params.update({
        'scale_pos_weight': scale_pos_weight,
        'random_state': RANDOM_SEED,
        'verbose': -1,
        'n_jobs': -1,
    })
    final_model = lgb.LGBMClassifier(**best_params)
    model_label = 'LightGBM (Tuned)'

# Train on full training data
final_model.fit(X_train, y_train)

# Validate
y_val_pred_final = final_model.predict(X_val)
y_val_prob_final = final_model.predict_proba(X_val)[:, 1]

print(f"\n{model_label} — Validation Performance:")
print(f"  Accuracy:  {accuracy_score(y_val, y_val_pred_final):.4f}")
print(f"  Precision: {precision_score(y_val, y_val_pred_final):.4f}")
print(f"  Recall:    {recall_score(y_val, y_val_pred_final):.4f}")
print(f"  F1:        {f1_score(y_val, y_val_pred_final):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_val, y_val_prob_final):.4f}")
print(f"  PR-AUC:    {average_precision_score(y_val, y_val_prob_final):.4f}")

# Compare with untuned
if best_model_name in results:
    print(f"\nImprovement over untuned {best_model_name}:")
    print(f"  F1:      {f1_score(y_val, y_val_pred_final) - results[best_model_name]['F1']:+.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_val, y_val_prob_final) - results[best_model_name]['ROC-AUC']:+.4f}")


LightGBM (Tuned) — Validation Performance:
  Accuracy:  0.9424
  Precision: 0.8621
  Recall:    0.8879
  F1:        0.8748
  ROC-AUC:   0.9828
  PR-AUC:    0.9440

Improvement over untuned LightGBM:
  F1:      +0.0703
  ROC-AUC: +0.0161


## 12. Model Evaluation on Test Set

Final evaluation on the **untouched test set** using the tuned model. This is the definitive performance assessment.

In [33]:
# ============================================================
# 12.1 Test set predictions
# ============================================================
y_test_pred = final_model.predict(X_test)
y_test_prob = final_model.predict_proba(X_test)[:, 1]

print("=" * 60)
print(f"FINAL MODEL EVALUATION — {model_label}")
print("=" * 60)
print(f"\nTest Set Size: {len(y_test):,}")
print(f"Positive samples in test: {y_test.sum():,}")
print()

# All metrics
test_metrics = {
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1-Score': f1_score(y_test, y_test_pred),
    'ROC-AUC': roc_auc_score(y_test, y_test_prob),
    'PR-AUC': average_precision_score(y_test, y_test_prob),
}

for metric, value in test_metrics.items():
    print(f"  {metric:12s}: {value:.4f}")

print(f"\n\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Stable', 'Landslide']))

FINAL MODEL EVALUATION — LightGBM (Tuned)

Test Set Size: 37,500
Positive samples in test: 8,504

  Accuracy    : 0.9438
  Precision   : 0.8627
  Recall      : 0.8944
  F1-Score    : 0.8782
  ROC-AUC     : 0.9835
  PR-AUC      : 0.9431


Classification Report:
              precision    recall  f1-score   support

      Stable       0.97      0.96      0.96     28996
   Landslide       0.86      0.89      0.88      8504

    accuracy                           0.94     37500
   macro avg       0.92      0.93      0.92     37500
weighted avg       0.94      0.94      0.94     37500



In [34]:
# ============================================================
# 12.2 Generalization check (train vs val vs test)
# ============================================================
y_train_pred = final_model.predict(X_train)
y_train_prob = final_model.predict_proba(X_train)[:, 1]

print("Generalization Check — F1 Score across splits:")
print(f"  Train: {f1_score(y_train, y_train_pred):.4f}")
print(f"  Val:   {f1_score(y_val, y_val_pred_final):.4f}")
print(f"  Test:  {f1_score(y_test, y_test_pred):.4f}")
print()

train_test_gap = f1_score(y_train, y_train_pred) - f1_score(y_test, y_test_pred)
print(f"Train-Test F1 gap: {train_test_gap:.4f}")
if train_test_gap > 0.1:
    print("⚠️ Significant gap detected — model may be overfitting")
elif train_test_gap > 0.05:
    print("⚠  Moderate gap — acceptable but worth monitoring")
else:
    print("✅ Good generalization — minimal overfitting detected")

Generalization Check — F1 Score across splits:
  Train: 1.0000
  Val:   0.8748
  Test:  0.8782

Train-Test F1 gap: 0.1218
⚠️ Significant gap detected — model may be overfitting


## 13. Model Explainability

Understanding what the model learned through feature importance analysis and error examination.

> **Important**: Feature importance shows which features the model relies on for predictions. This reflects **correlation and predictive power**, not necessarily **causal relationships**.

In [35]:
# ============================================================
# 13.1 Built-in Feature Importance
# ============================================================
if hasattr(final_model, 'feature_importances_'):
    importance = pd.Series(
        final_model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)
    
    print("Top 15 Features by Built-in Importance:")
    print("-" * 45)
    for i, (feat, imp) in enumerate(importance.head(15).items()):
        bar = "█" * int(imp / importance.max() * 30)
        print(f"  {i+1:2d}. {feat:30s} {imp:.4f}  {bar}")
else:
    print("Model does not have built-in feature_importances_")

Top 15 Features by Built-in Importance:
---------------------------------------------
   1. aspect                         5395.0000  ██████████████████████████████
   2. elevation                      4411.0000  ████████████████████████
   3. GLCMMean                       3929.0000  █████████████████████
   4. mNDWI                          3738.0000  ████████████████████
   5. profile_curvature              3129.0000  █████████████████
   6. plan_curvature                 3111.0000  █████████████████
   7. EVI                            3053.0000  ████████████████
   8. GLCMCorrelation                2788.0000  ███████████████
   9. NDTI                           2426.0000  █████████████
  10. ARVI                           2397.0000  █████████████
  11. slope_ndvi_interaction         2343.0000  █████████████
  12. twi                            2217.0000  ████████████
  13. GRVI                           2216.0000  ████████████
  14. curvature                      2199.0000  ██████

In [36]:
# ============================================================
# 13.2 Permutation Importance (on validation set)
# ============================================================
print("Calculating permutation importance (this may take a moment)...")

perm_imp = permutation_importance(
    final_model, X_val, y_val, 
    n_repeats=10, random_state=RANDOM_SEED, 
    scoring='f1', n_jobs=-1
)

perm_importance = pd.Series(
    perm_imp.importances_mean,
    index=X_val.columns
).sort_values(ascending=False)

print("\nTop 15 Features by Permutation Importance (F1 decrease):")
print("-" * 50)
for i, (feat, imp) in enumerate(perm_importance.head(15).items()):
    std = perm_imp.importances_std[X_val.columns.get_loc(feat)]
    print(f"  {i+1:2d}. {feat:30s} {imp:.4f} ± {std:.4f}")

Calculating permutation importance (this may take a moment)...



Top 15 Features by Permutation Importance (F1 decrease):
--------------------------------------------------
   1. GLCMMean                       0.5648 ± 0.0049
   2. elevation                      0.5611 ± 0.0065
   3. ARVI                           0.1233 ± 0.0014
   4. aspect                         0.1202 ± 0.0020
   5. mNDWI                          0.1073 ± 0.0016
   6. EVI                            0.0642 ± 0.0019
   7. NDVI                           0.0539 ± 0.0017
   8. NDTI                           0.0505 ± 0.0017
   9. GLCMCorrelation                0.0484 ± 0.0027
  10. vegetation_stress              0.0327 ± 0.0013
  11. SAVI                           0.0311 ± 0.0012
  12. GRVI                           0.0255 ± 0.0013
  13. MSAVI                          0.0186 ± 0.0013
  14. mNDMI                          0.0185 ± 0.0012
  15. BSI                            0.0172 ± 0.0010


In [37]:
# ============================================================
# 13.3 Error Analysis
# ============================================================
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# Confusion matrix breakdown
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives  (correctly predicted stable):    {tn:,}")
print(f"  True Positives  (correctly predicted landslide): {tp:,}")
print(f"  False Positives (false alarm):                   {fp:,}")
print(f"  False Negatives (missed landslide):              {fn:,}")
print()

print(f"False Positive Rate: {fp/(fp+tn):.4f} ({fp/(fp+tn)*100:.2f}%)")
print(f"False Negative Rate: {fn/(fn+tp):.4f} ({fn/(fn+tp)*100:.2f}%)")
print()

# Analyze characteristics of errors
test_df = X_test.copy()
test_df['true_label'] = y_test.values
test_df['predicted'] = y_test_pred
test_df['pred_prob'] = y_test_prob

# False negatives analysis (missed landslides — most critical)
fn_mask = (test_df['true_label'] == 1) & (test_df['predicted'] == 0)
fp_mask = (test_df['true_label'] == 0) & (test_df['predicted'] == 1)

if fn_mask.sum() > 0:
    print(f"False Negative Analysis (missed landslides, n={fn_mask.sum()}):")
    fn_data = test_df[fn_mask]
    top_feats = importance.head(5).index.tolist() if hasattr(final_model, 'feature_importances_') else ALL_FEATURES[:5]
    for feat in top_feats:
        fn_mean = fn_data[feat].mean()
        tp_mean = test_df[(test_df['true_label']==1) & (test_df['predicted']==1)][feat].mean()
        print(f"  {feat:25s}: FN mean={fn_mean:.4f}, TP mean={tp_mean:.4f}, diff={fn_mean-tp_mean:+.4f}")

    print(f"\nFalse Negative prediction probability:")
    print(f"  Mean:   {fn_data['pred_prob'].mean():.4f}")
    print(f"  Median: {fn_data['pred_prob'].median():.4f}")
    print(f"  Max:    {fn_data['pred_prob'].max():.4f}")

ERROR ANALYSIS

Confusion Matrix Breakdown:
  True Negatives  (correctly predicted stable):    27,785
  True Positives  (correctly predicted landslide): 7,606
  False Positives (false alarm):                   1,211
  False Negatives (missed landslide):              898

False Positive Rate: 0.0418 (4.18%)
False Negative Rate: 0.1056 (10.56%)

False Negative Analysis (missed landslides, n=898):
  aspect                   : FN mean=208.1902, TP mean=181.0974, diff=+27.0928
  elevation                : FN mean=1519.5038, TP mean=1452.0299, diff=+67.4739
  GLCMMean                 : FN mean=26.1062, TP mean=24.0973, diff=+2.0089
  mNDWI                    : FN mean=-0.1778, TP mean=-0.2070, diff=+0.0292
  profile_curvature        : FN mean=-0.0002, TP mean=-0.0002, diff=+0.0000

False Negative prediction probability:
  Mean:   0.2295
  Median: 0.2262
  Max:    0.4997


## 14. Final Visualizations

Publication-quality visualizations of model performance, saved to `output/`.

In [38]:
# ============================================================
# 14.1 Confusion Matrix
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, display_labels=['Stable', 'Landslide'],
    cmap='Blues', values_format=',d', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, display_labels=['Stable', 'Landslide'],
    cmap='Blues', normalize='true', values_format='.2%', ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'{model_label} — Test Set Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'confusion_matrix.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\confusion_matrix.png


In [39]:
# ============================================================
# 14.2 ROC Curve (all models)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 7))

# Plot ROC for all models
model_predictions = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        y_prob_m = model.predict_proba(X_val_scaled)[:, 1]
    else:
        y_prob_m = model.predict_proba(X_val)[:, 1]
    model_predictions[name] = y_prob_m
    
    fpr, tpr, _ = roc_curve(y_val, y_prob_m)
    auc = roc_auc_score(y_val, y_prob_m)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

# Tuned model
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
auc = roc_auc_score(y_test, y_test_prob)
ax.plot(fpr, tpr, label=f'{model_label} — Test (AUC={auc:.4f})', linewidth=2.5, linestyle='--')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'roc_curve.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\roc_curve.png


In [40]:
# ============================================================
# 14.3 Precision-Recall Curve
# ============================================================
fig, ax = plt.subplots(figsize=(8, 7))

for name, y_prob_m in model_predictions.items():
    prec, rec, _ = precision_recall_curve(y_val, y_prob_m)
    ap = average_precision_score(y_val, y_prob_m)
    ax.plot(rec, prec, label=f'{name} (AP={ap:.4f})', linewidth=2)

# Tuned model on test
prec, rec, _ = precision_recall_curve(y_test, y_test_prob)
ap = average_precision_score(y_test, y_test_prob)
ax.plot(rec, prec, label=f'{model_label} — Test (AP={ap:.4f})', linewidth=2.5, linestyle='--')

# Baseline (proportion of positive class)
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'precision_recall_curve.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'precision_recall_curve.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\precision_recall_curve.png


In [41]:
# ============================================================
# 14.4 Feature Importance Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Built-in importance
if hasattr(final_model, 'feature_importances_'):
    top_n = 15
    top_imp = importance.head(top_n)
    
    colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
    axes[0].barh(range(top_n), top_imp.values[::-1], color=colors_imp)
    axes[0].set_yticks(range(top_n))
    axes[0].set_yticklabels(top_imp.index[::-1])
    axes[0].set_title(f'{model_label}\nBuilt-in Feature Importance', fontweight='bold')
    axes[0].set_xlabel('Importance Score')
    
    for i, v in enumerate(top_imp.values[::-1]):
        axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

# Permutation importance
top_perm = perm_importance.head(top_n)
colors_perm = plt.cm.magma(np.linspace(0.3, 0.9, top_n))
axes[1].barh(range(top_n), top_perm.values[::-1], color=colors_perm)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(top_perm.index[::-1])
axes[1].set_title(f'{model_label}\nPermutation Importance (F1)', fontweight='bold')
axes[1].set_xlabel('Mean F1 Decrease')

for i, v in enumerate(top_perm.values[::-1]):
    axes[1].text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Feature Importance Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'feature_importance.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\feature_importance.png


In [42]:
# ============================================================
# 14.5 Prediction Probability Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution by true class
for label, color, name in [(0, COLORS['negative'], 'Stable'), 
                            (1, COLORS['positive'], 'Landslide')]:
    mask = y_test == label
    axes[0].hist(y_test_prob[mask], bins=50, alpha=0.6, label=name, color=color, density=True)
axes[0].set_xlabel('Predicted Probability of Landslide')
axes[0].set_ylabel('Density')
axes[0].set_title('Prediction Probability by True Class', fontweight='bold')
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Decision Threshold (0.5)')
axes[0].legend()

# Calibration-style plot
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, y_test_prob, n_bins=15, strategy='quantile')
axes[1].plot(prob_pred, prob_true, 's-', color=COLORS['primary'], label='Model', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly Calibrated')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prediction_distribution.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'prediction_distribution.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\prediction_distribution.png


In [43]:
# ============================================================
# 14.6 Model Performance Summary Card
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# Create summary table
summary_data = [
    ['Metric', 'Value'],
    ['Model', model_label],
    ['Accuracy', f"{test_metrics['Accuracy']:.4f}"],
    ['Precision', f"{test_metrics['Precision']:.4f}"],
    ['Recall', f"{test_metrics['Recall']:.4f}"],
    ['F1-Score', f"{test_metrics['F1-Score']:.4f}"],
    ['ROC-AUC', f"{test_metrics['ROC-AUC']:.4f}"],
    ['PR-AUC', f"{test_metrics['PR-AUC']:.4f}"],
    ['', ''],
    ['Train-Test F1 Gap', f"{train_test_gap:.4f}"],
    ['Test Samples', f"{len(y_test):,}"],
    ['Features Used', f"{len(X_train.columns)}"],
]

table = ax.table(cellText=summary_data, loc='center', cellLoc='center',
                 colWidths=[0.35, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)

# Style header
for j in range(2):
    table[0, j].set_facecolor('#2E86AB')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Alternate row colors
for i in range(1, len(summary_data)):
    for j in range(2):
        if i % 2 == 0:
            table[i, j].set_facecolor('#f0f0f0')

ax.set_title(f'Final Model Performance Summary\n{model_label}', 
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_performance.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'model_performance.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\model_performance.png


In [44]:
# ============================================================
# 14.7 Final Model Comparison Chart
# ============================================================
# Add tuned model to results
results[model_label] = {
    'Accuracy': test_metrics['Accuracy'],
    'Precision': test_metrics['Precision'],
    'Recall': test_metrics['Recall'],
    'F1': test_metrics['F1-Score'],
    'ROC-AUC': test_metrics['ROC-AUC'],
    'PR-AUC': test_metrics['PR-AUC'],
}

final_results_df = pd.DataFrame(results).T.sort_values('F1', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

metrics_plot = ['F1', 'ROC-AUC', 'PR-AUC', 'Precision', 'Recall']
x = np.arange(len(final_results_df))
width = 0.15
palette = sns.color_palette('husl', len(metrics_plot))

for i, (metric, color) in enumerate(zip(metrics_plot, palette)):
    bars = ax.bar(x + i * width, final_results_df[metric], width, 
                  label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(final_results_df.index, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Complete Model Comparison (All Metrics)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_comparison.png')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'model_comparison.png'}")

Saved: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\model_comparison.png


## 15. Conclusions and Recommendations

In [45]:
# ============================================================
# 15.1 Save final results summary
# ============================================================
summary = {
    'project': 'Sikkim Landslide Prediction',
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'total_rows': 4_000_000,
        'subsample_size': len(df),
        'features': len(ALL_FEATURES_ENGINEERED),
        'positive_class_ratio': float(y.mean()),
    },
    'best_model': model_label,
    'best_params': study.best_params,
    'test_metrics': test_metrics,
    'top_features': importance.head(10).to_dict() if hasattr(final_model, 'feature_importances_') else {},
    'generalization': {
        'train_f1': float(f1_score(y_train, y_train_pred)),
        'val_f1': float(f1_score(y_val, y_val_pred_final)),
        'test_f1': float(f1_score(y_test, y_test_pred)),
        'train_test_gap': float(train_test_gap),
    }
}

with open(OUTPUT_DIR / 'results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Results saved to: {OUTPUT_DIR / 'results_summary.json'}")

Results saved to: C:\Users\AGNIV GHOSH\OneDrive\Desktop\LandslidePredictionSoftware\output\results_summary.json


### Summary

#### Problem Definition
- **Task**: Binary classification for landslide susceptibility mapping in Sikkim, India
- **Challenge**: Severe class imbalance (~1.5% positive class)
- **Features**: 30 geospatial features from Sentinel-1/2 + 4 engineered features

#### Dataset Characteristics
- ~4 million pixel-level observations from satellite imagery
- 12 spectral indices, 8 topographic parameters, 2 hydrological features, 8 textural metrics
- Only GLCMCorrelation had missing values (~0.9%)
- Several features had extreme outliers (EVI, SPI)

#### Preprocessing Performed
- Stratified subsampling (retaining all positive class samples)
- Median imputation for missing values (fitted on training data)
- IQR-based outlier clipping (1st-99th percentile, fitted on training data)
- StandardScaler normalization (fitted on training data)
- SMOTE oversampling on training data for the Logistic Regression baseline
- All transformers fitted exclusively on training data to prevent leakage

#### Models Evaluated
1. **Logistic Regression** — Interpretable baseline
2. **Random Forest** — Ensemble with feature importance
3. **XGBoost** — Gradient boosting with regularization
4. **LightGBM** — Fast gradient boosting

#### Key Findings
- Gradient boosting models (XGBoost/LightGBM) significantly outperformed the linear baseline
- The model effectively distinguishes landslide-prone areas from stable terrain
- Top predictive features align with domain knowledge (topographic and vegetation indices)
- Feature importance highlights correlation, not causation

#### Limitations
1. **Spatial autocorrelation**: Adjacent pixels are not independent — performance may be optimistic
2. **Temporal generalization**: Model trained on specific events may not generalize to future conditions
3. **Class definition**: Binary decision boundary may oversimplify gradual susceptibility
4. **Subsampling**: Results based on stratified subsample, not full 4M dataset

#### Recommendations for Future Improvement
1. **Spatial cross-validation**: Split by geographic region to assess true generalization
2. **Temporal validation**: Test on events from different time periods
3. **Threshold optimization**: Tune decision threshold based on cost of false negatives vs. false positives
4. **Additional features**: Incorporate rainfall data, geological maps, land-use classification
5. **Ensemble methods**: Stack multiple models for improved robustness
6. **Full dataset training**: With sufficient compute, train on complete 4M-row dataset

In [46]:
# ============================================================
# 15.2 List all generated artifacts
# ============================================================
print("=" * 60)
print("GENERATED ARTIFACTS")
print("=" * 60)

for directory, label in [(ARTIFACTS_DIR, 'artifacts/'), (OUTPUT_DIR, 'output/')]:
    print(f"\n{label}")
    print("-" * 40)
    for f in sorted(directory.glob('*')):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s} ({size_kb:.1f} KB)")

print("\n✅ Notebook execution complete!")

GENERATED ARTIFACTS

artifacts/
----------------------------------------
  correlation_matrix.png                   (191.8 KB)
  feature_category_analysis.png            (66.2 KB)
  feature_distributions_hydrological.png   (51.0 KB)
  feature_distributions_spectral.png       (229.5 KB)
  feature_distributions_textural.png       (138.2 KB)
  feature_distributions_topographical.png  (183.0 KB)
  feature_target_relationships.png         (167.5 KB)
  missing_values.png                       (48.8 KB)
  model_comparison_validation.png          (105.4 KB)
  outlier_analysis.png                     (143.5 KB)
  target_distribution.png                  (100.6 KB)

output/
----------------------------------------
  confusion_matrix.png                     (80.6 KB)
  feature_importance.png                   (183.7 KB)
  model_comparison.png                     (72.7 KB)
  model_performance.png                    (77.2 KB)
  precision_recall_curve.png               (132.3 KB)
  prediction_distri